# QuoteGuard — Evaluation Data Generator

## Purpose

This notebook generates the synthetic supplier quotation dataset used for the **QuoteGuard** project.

The dataset is designed to represent realistic procurement scenarios, including valid quotations, quotations requiring managerial review, quotations containing unclear or inconsistent information, and invalid quotations that should not proceed further.

A total of **50 quotation cases** are generated with predefined ground-truth outcomes:

| Decision | Cases | Description |
|---|---:|---|
| APPROVE | 25 | Valid and complete quotations suitable for further procurement processing |
| REVIEW | 10 | Quotations containing commercial conditions that require supervisor judgement |
| REQUEST_CLARIFICATION | 7 | Quotations containing missing, unclear, contradictory, or incorrectly calculated information |
| REJECT | 8 | Invalid or unsuitable quotations that should not proceed |
| **Total** | **50** | |

## Dataset Structure

Each evaluation case contains three main components:

- `eval_id` — Unique identifier in the format `PQE-2026-XXX`
- `quotation` — Structured supplier quotation containing the information presented for assessment
- `expected` — Predefined ground-truth outcome for the quotation
- `tags` — Scenario labels used to identify the characteristics of each case

### Quotation Data

The `quotation` object contains relevant procurement information such as:

- Supplier details
- Quotation number and dates
- Currency
- Products and quantities
- Unit and total prices
- Discounts and taxes
- Delivery terms
- Warranty
- Payment terms
- Other relevant commercial conditions

### Ground Truth

The `expected` object contains:

- `decision` — APPROVE, REVIEW, REQUEST_CLARIFICATION, or REJECT
- `reason` — Reason for the expected outcome
- `issue` — Primary issue associated with the quotation
- `next_action` — Expected procurement workflow action
- `severity` — Severity of the identified issue

## Output

The generated dataset is stored as:

`data/data.json`

This file will serve as the controlled input and ground-truth dataset for the QuoteGuard agent developed separately.

In [1]:
# ==========================================
# CELL 1: DRIVE & PROJECT CONFIGURATION
# ==========================================

from google.colab import drive
from pathlib import Path

# Mount Google Drive
drive.mount("/content/drive")

# Project directory
PROJECT_DIR = Path("/content/drive/MyDrive/QuoteGuard - AI Procurement")

# Data directory
DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Final generated evaluation dataset
DATA_FILE = DATA_DIR / "data.json"

print("QuoteGuard project initialized")
print(f"Project directory : {PROJECT_DIR}")
print(f"Output file       : {DATA_FILE}")

Mounted at /content/drive
QuoteGuard project initialized
Project directory : /content/drive/MyDrive/QuoteGuard - AI Procurement
Output file       : /content/drive/MyDrive/QuoteGuard - AI Procurement/data/data.json


In [3]:
# ==========================================
# CELL 2: IMPORTS & REPRODUCIBILITY
# ==========================================

import json
import random
from datetime import datetime, timedelta
from decimal import Decimal, ROUND_HALF_UP

# Fixed seed so the generated dataset is reproducible
random.seed(42)

# Reference date used when generating quotation dates
REFERENCE_DATE = datetime(2026, 9, 30)

print("Generator utilities loaded.")

Generator utilities loaded.


In [4]:
# ==========================================
# CELL 3: PROCUREMENT CONTEXT & MASTER DATA
# ==========================================

# QuoteGuard procurement policy used across the synthetic cases
PROCUREMENT_POLICY = {
    "base_currency": "SGD",
    "standard_tax_rate": 9.0,
    "supervisor_review_threshold": 100000.00,
    "preferred_max_delivery_days": 30,
    "preferred_min_warranty_months": 12
}

# Synthetic suppliers
SUPPLIERS = [
    "Apex Technology Pte. Ltd.",
    "Nova Business Systems Pte. Ltd.",
    "Vertex Enterprise Supplies Pte. Ltd.",
    "Orion Office Technologies Pte. Ltd.",
    "Summit Digital Solutions Pte. Ltd.",
    "Nexus Technology Pte. Ltd.",
    "PrimeTech Distribution Pte. Ltd.",
    "Horizon Enterprise Systems Pte. Ltd."
]

# Products commonly used in the synthetic quotations
PRODUCTS = [
    {
        "name": "Business Laptop",
        "model": "BL-14X",
        "price_range": (1100, 1800)
    },
    {
        "name": "27-inch Business Monitor",
        "model": "BM-27Q",
        "price_range": (250, 550)
    },
    {
        "name": "USB-C Docking Station",
        "model": "DS-100",
        "price_range": (120, 280)
    },
    {
        "name": "Wireless Keyboard and Mouse Set",
        "model": "KM-500",
        "price_range": (50, 150)
    },
    {
        "name": "Enterprise Wi-Fi Access Point",
        "model": "AP-600",
        "price_range": (350, 900)
    }
]

print("Procurement context initialized.")
print(f"Base currency              : {PROCUREMENT_POLICY['base_currency']}")
print(f"Standard tax rate          : {PROCUREMENT_POLICY['standard_tax_rate']}%")
print(f"Supervisor review threshold: SGD {PROCUREMENT_POLICY['supervisor_review_threshold']:,.2f}")
print(f"Synthetic suppliers        : {len(SUPPLIERS)}")
print(f"Product types              : {len(PRODUCTS)}")

Procurement context initialized.
Base currency              : SGD
Standard tax rate          : 9.0%
Supervisor review threshold: SGD 100,000.00
Synthetic suppliers        : 8
Product types              : 5


In [5]:
# ==========================================
# CELL 4: APPROVE QUOTATION CASES
# 25 valid quotations expected to proceed
# ==========================================

approve_cases = [

    # ==========================================================
    # PQE-2026-001 — Standard single-item quotation
    # ==========================================================
    {
        "eval_id": "PQE-2026-001",
        "quotation": {
            "supplier_name": "Apex Technology Pte. Ltd.",
            "quotation_number": "QT-2026-1001",
            "quotation_date": "2026-09-20",
            "valid_until": "2026-10-20",
            "currency": "SGD",
            "items": [
                {
                    "description": "Business Laptop",
                    "model": "BL-14X",
                    "quantity": 20,
                    "unit_price": 1250.00,
                    "line_total": 25000.00
                }
            ],
            "subtotal": 25000.00,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 9.0,
            "tax_amount": 2250.00,
            "shipping_cost": 0.00,
            "additional_charges": 0.00,
            "total_amount": 27250.00,
            "delivery_terms": "Delivery within 14 days",
            "warranty": "24 months",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "APPROVE",
            "reason": "Quotation is complete, commercially acceptable and internally consistent.",
            "issue": "NONE",
            "next_action": "CREATE_DRAFT_PO",
            "severity": "NONE"
        },
        "tags": ["standard_quotation", "single_item", "valid_calculation"]
    },

    # ==========================================================
    # PQE-2026-002 — Multiple-item quotation
    # ==========================================================
    {
        "eval_id": "PQE-2026-002",
        "quotation": {
            "supplier_name": "Nova Business Systems Pte. Ltd.",
            "quotation_number": "QT-2026-1002",
            "quotation_date": "2026-09-21",
            "valid_until": "2026-10-21",
            "currency": "SGD",
            "items": [
                {
                    "description": "27-inch Business Monitor",
                    "model": "BM-27Q",
                    "quantity": 20,
                    "unit_price": 400.00,
                    "line_total": 8000.00
                },
                {
                    "description": "USB-C Docking Station",
                    "model": "DS-100",
                    "quantity": 20,
                    "unit_price": 180.00,
                    "line_total": 3600.00
                }
            ],
            "subtotal": 11600.00,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 9.0,
            "tax_amount": 1044.00,
            "shipping_cost": 100.00,
            "additional_charges": 0.00,
            "total_amount": 12744.00,
            "delivery_terms": "Delivery within 10 working days",
            "warranty": "24 months",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "APPROVE",
            "reason": "Quotation contains complete product and commercial information with correct calculations.",
            "issue": "NONE",
            "next_action": "CREATE_DRAFT_PO",
            "severity": "NONE"
        },
        "tags": ["multiple_items", "shipping", "valid_calculation"]
    },

    # ==========================================================
    # PQE-2026-003 — Valid 10% discount
    # ==========================================================
    {
        "eval_id": "PQE-2026-003",
        "quotation": {
            "supplier_name": "Vertex Enterprise Supplies Pte. Ltd.",
            "quotation_number": "QT-2026-1003",
            "quotation_date": "2026-09-22",
            "valid_until": "2026-10-22",
            "currency": "SGD",
            "items": [
                {
                    "description": "Enterprise Wi-Fi Access Point",
                    "model": "AP-600",
                    "quantity": 10,
                    "unit_price": 600.00,
                    "line_total": 6000.00
                }
            ],
            "subtotal": 6000.00,
            "discount_percent": 10,
            "discount_amount": 600.00,
            "tax_percent": 9.0,
            "tax_amount": 486.00,
            "shipping_cost": 0.00,
            "additional_charges": 0.00,
            "total_amount": 5886.00,
            "delivery_terms": "Delivery within 7 days",
            "warranty": "36 months",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "APPROVE",
            "reason": "Quotation is complete and the discount, tax and final amount are calculated correctly.",
            "issue": "NONE",
            "next_action": "CREATE_DRAFT_PO",
            "severity": "NONE"
        },
        "tags": ["discount", "valid_tax", "valid_calculation"]
    },

    # ==========================================================
    # PQE-2026-004 — Valid 5% volume discount
    # ==========================================================
    {
        "eval_id": "PQE-2026-004",
        "quotation": {
            "supplier_name": "Orion Office Technologies Pte. Ltd.",
            "quotation_number": "QT-2026-1004",
            "quotation_date": "2026-09-18",
            "valid_until": "2026-10-18",
            "currency": "SGD",
            "items": [
                {
                    "description": "Wireless Keyboard and Mouse Set",
                    "model": "KM-500",
                    "quantity": 100,
                    "unit_price": 80.00,
                    "line_total": 8000.00
                }
            ],
            "subtotal": 8000.00,
            "discount_percent": 5,
            "discount_amount": 400.00,
            "tax_percent": 9.0,
            "tax_amount": 684.00,
            "shipping_cost": 0.00,
            "additional_charges": 0.00,
            "total_amount": 8284.00,
            "delivery_terms": "Delivery within 14 days",
            "warranty": "12 months",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "APPROVE",
            "reason": "Volume discount and all resulting totals are correctly calculated.",
            "issue": "NONE",
            "next_action": "CREATE_DRAFT_PO",
            "severity": "NONE"
        },
        "tags": ["volume_discount", "large_quantity", "valid_calculation"]
    },

    # ==========================================================
    # PQE-2026-005 — Decimal unit prices
    # ==========================================================
    {
        "eval_id": "PQE-2026-005",
        "quotation": {
            "supplier_name": "Summit Digital Solutions Pte. Ltd.",
            "quotation_number": "QT-2026-1005",
            "quotation_date": "2026-09-19",
            "valid_until": "2026-10-19",
            "currency": "SGD",
            "items": [
                {
                    "description": "USB-C Docking Station",
                    "model": "DS-100",
                    "quantity": 15,
                    "unit_price": 189.50,
                    "line_total": 2842.50
                }
            ],
            "subtotal": 2842.50,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 9.0,
            "tax_amount": 255.83,
            "shipping_cost": 50.00,
            "additional_charges": 0.00,
            "total_amount": 3148.33,
            "delivery_terms": "Delivery within 7 days",
            "warranty": "24 months",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "APPROVE",
            "reason": "Decimal pricing, tax and final payable amount are correctly calculated.",
            "issue": "NONE",
            "next_action": "CREATE_DRAFT_PO",
            "severity": "NONE"
        },
        "tags": ["decimal_price", "rounding", "valid_calculation"]
    },

    # ==========================================================
    # PQE-2026-006 — Large quantity with correct calculations
    # ==========================================================
    {
        "eval_id": "PQE-2026-006",
        "quotation": {
            "supplier_name": "Nexus Technology Pte. Ltd.",
            "quotation_number": "QT-2026-1006",
            "quotation_date": "2026-09-20",
            "valid_until": "2026-10-20",
            "currency": "SGD",
            "items": [
                {
                    "description": "Wireless Keyboard and Mouse Set",
                    "model": "KM-500",
                    "quantity": 500,
                    "unit_price": 60.00,
                    "line_total": 30000.00
                }
            ],
            "subtotal": 30000.00,
            "discount_percent": 10,
            "discount_amount": 3000.00,
            "tax_percent": 9.0,
            "tax_amount": 2430.00,
            "shipping_cost": 0.00,
            "additional_charges": 0.00,
            "total_amount": 29430.00,
            "delivery_terms": "Delivery within 21 days",
            "warranty": "12 months",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "APPROVE",
            "reason": "Large quantity quotation remains within policy limits and all calculations are correct.",
            "issue": "NONE",
            "next_action": "CREATE_DRAFT_PO",
            "severity": "NONE"
        },
        "tags": ["large_quantity", "volume_discount", "valid_calculation"]
    },

    # ==========================================================
    # PQE-2026-007 — Shipping charge included correctly
    # ==========================================================
    {
        "eval_id": "PQE-2026-007",
        "quotation": {
            "supplier_name": "PrimeTech Distribution Pte. Ltd.",
            "quotation_number": "QT-2026-1007",
            "quotation_date": "2026-09-23",
            "valid_until": "2026-10-23",
            "currency": "SGD",
            "items": [
                {
                    "description": "27-inch Business Monitor",
                    "model": "BM-27Q",
                    "quantity": 30,
                    "unit_price": 350.00,
                    "line_total": 10500.00
                }
            ],
            "subtotal": 10500.00,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 9.0,
            "tax_amount": 945.00,
            "shipping_cost": 200.00,
            "additional_charges": 0.00,
            "total_amount": 11645.00,
            "delivery_terms": "Delivery within 10 days",
            "warranty": "24 months",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "APPROVE",
            "reason": "Shipping charge is clearly disclosed and correctly included in the final amount.",
            "issue": "NONE",
            "next_action": "CREATE_DRAFT_PO",
            "severity": "NONE"
        },
        "tags": ["shipping", "single_item", "valid_calculation"]
    },

    # ==========================================================
    # PQE-2026-008 — Free delivery
    # ==========================================================
    {
        "eval_id": "PQE-2026-008",
        "quotation": {
            "supplier_name": "Horizon Enterprise Systems Pte. Ltd.",
            "quotation_number": "QT-2026-1008",
            "quotation_date": "2026-09-24",
            "valid_until": "2026-10-24",
            "currency": "SGD",
            "items": [
                {
                    "description": "Business Laptop",
                    "model": "BL-14X",
                    "quantity": 10,
                    "unit_price": 1400.00,
                    "line_total": 14000.00
                }
            ],
            "subtotal": 14000.00,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 9.0,
            "tax_amount": 1260.00,
            "shipping_cost": 0.00,
            "additional_charges": 0.00,
            "total_amount": 15260.00,
            "delivery_terms": "Free delivery within 14 days",
            "warranty": "24 months",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "APPROVE",
            "reason": "Quotation is complete and free delivery is explicitly stated.",
            "issue": "NONE",
            "next_action": "CREATE_DRAFT_PO",
            "severity": "NONE"
        },
        "tags": ["free_shipping", "valid_terms"]
    },

    # ==========================================================
    # PQE-2026-009 — Installation charge
    # ==========================================================
    {
        "eval_id": "PQE-2026-009",
        "quotation": {
            "supplier_name": "Apex Technology Pte. Ltd.",
            "quotation_number": "QT-2026-1009",
            "quotation_date": "2026-09-17",
            "valid_until": "2026-10-17",
            "currency": "SGD",
            "items": [
                {
                    "description": "Enterprise Wi-Fi Access Point",
                    "model": "AP-600",
                    "quantity": 20,
                    "unit_price": 500.00,
                    "line_total": 10000.00
                }
            ],
            "subtotal": 10000.00,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 9.0,
            "tax_amount": 900.00,
            "shipping_cost": 0.00,
            "additional_charges": 500.00,
            "total_amount": 11400.00,
            "delivery_terms": "Delivery and installation within 14 days",
            "warranty": "36 months",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "APPROVE",
            "reason": "Installation charge is clearly disclosed and correctly included in the final amount.",
            "issue": "NONE",
            "next_action": "CREATE_DRAFT_PO",
            "severity": "NONE"
        },
        "tags": ["installation_charge", "additional_charge", "valid_calculation"]
    },

    # ==========================================================
    # PQE-2026-010 — Delivery expressed in working days
    # ==========================================================
    {
        "eval_id": "PQE-2026-010",
        "quotation": {
            "supplier_name": "Nova Business Systems Pte. Ltd.",
            "quotation_number": "QT-2026-1010",
            "quotation_date": "2026-09-22",
            "valid_until": "2026-10-22",
            "currency": "SGD",
            "items": [
                {
                    "description": "Business Laptop",
                    "model": "BL-14X",
                    "quantity": 15,
                    "unit_price": 1300.00,
                    "line_total": 19500.00
                }
            ],
            "subtotal": 19500.00,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 9.0,
            "tax_amount": 1755.00,
            "shipping_cost": 0.00,
            "additional_charges": 0.00,
            "total_amount": 21255.00,
            "delivery_terms": "Delivery within 10 working days",
            "warranty": "24 months",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "APPROVE",
            "reason": "Delivery terms are clearly specified and fall within the acceptable delivery period.",
            "issue": "NONE",
            "next_action": "CREATE_DRAFT_PO",
            "severity": "NONE"
        },
        "tags": ["working_days", "delivery_terms"]
    },

    # ==========================================================
    # PQE-2026-011 — Delivery expressed in weeks
    # ==========================================================
    {
        "eval_id": "PQE-2026-011",
        "quotation": {
            "supplier_name": "Vertex Enterprise Supplies Pte. Ltd.",
            "quotation_number": "QT-2026-1011",
            "quotation_date": "2026-09-20",
            "valid_until": "2026-10-20",
            "currency": "SGD",
            "items": [
                {
                    "description": "27-inch Business Monitor",
                    "model": "BM-27Q",
                    "quantity": 25,
                    "unit_price": 360.00,
                    "line_total": 9000.00
                }
            ],
            "subtotal": 9000.00,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 9.0,
            "tax_amount": 810.00,
            "shipping_cost": 0.00,
            "additional_charges": 0.00,
            "total_amount": 9810.00,
            "delivery_terms": "Delivery within 3 weeks",
            "warranty": "24 months",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "APPROVE",
            "reason": "Delivery period is clearly stated and remains within the acceptable procurement window.",
            "issue": "NONE",
            "next_action": "CREATE_DRAFT_PO",
            "severity": "NONE"
        },
        "tags": ["delivery_weeks", "delivery_terms"]
    },

    # ==========================================================
    # PQE-2026-012 — Specific delivery date
    # ==========================================================
    {
        "eval_id": "PQE-2026-012",
        "quotation": {
            "supplier_name": "Orion Office Technologies Pte. Ltd.",
            "quotation_number": "QT-2026-1012",
            "quotation_date": "2026-09-20",
            "valid_until": "2026-10-20",
            "currency": "SGD",
            "items": [
                {
                    "description": "USB-C Docking Station",
                    "model": "DS-100",
                    "quantity": 30,
                    "unit_price": 170.00,
                    "line_total": 5100.00
                }
            ],
            "subtotal": 5100.00,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 9.0,
            "tax_amount": 459.00,
            "shipping_cost": 0.00,
            "additional_charges": 0.00,
            "total_amount": 5559.00,
            "delivery_terms": "Delivery by 15 October 2026",
            "warranty": "24 months",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "APPROVE",
            "reason": "A clear and acceptable delivery date is provided with complete commercial terms.",
            "issue": "NONE",
            "next_action": "CREATE_DRAFT_PO",
            "severity": "NONE"
        },
        "tags": ["specific_delivery_date", "delivery_terms"]
    },

    # ==========================================================
    # PQE-2026-013 — Warranty expressed in years
    # ==========================================================
    {
        "eval_id": "PQE-2026-013",
        "quotation": {
            "supplier_name": "Summit Digital Solutions Pte. Ltd.",
            "quotation_number": "QT-2026-1013",
            "quotation_date": "2026-09-21",
            "valid_until": "2026-10-21",
            "currency": "SGD",
            "items": [
                {
                    "description": "Business Laptop",
                    "model": "BL-14X",
                    "quantity": 12,
                    "unit_price": 1500.00,
                    "line_total": 18000.00
                }
            ],
            "subtotal": 18000.00,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 9.0,
            "tax_amount": 1620.00,
            "shipping_cost": 0.00,
            "additional_charges": 0.00,
            "total_amount": 19620.00,
            "delivery_terms": "Delivery within 14 days",
            "warranty": "2 years",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "APPROVE",
            "reason": "Warranty is clearly stated and exceeds the minimum warranty requirement.",
            "issue": "NONE",
            "next_action": "CREATE_DRAFT_PO",
            "severity": "NONE"
        },
        "tags": ["warranty_years", "valid_warranty"]
    },

    # ==========================================================
    # PQE-2026-014 — Extended warranty
    # ==========================================================
    {
        "eval_id": "PQE-2026-014",
        "quotation": {
            "supplier_name": "Nexus Technology Pte. Ltd.",
            "quotation_number": "QT-2026-1014",
            "quotation_date": "2026-09-23",
            "valid_until": "2026-10-23",
            "currency": "SGD",
            "items": [
                {
                    "description": "Enterprise Wi-Fi Access Point",
                    "model": "AP-600",
                    "quantity": 15,
                    "unit_price": 550.00,
                    "line_total": 8250.00
                }
            ],
            "subtotal": 8250.00,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 9.0,
            "tax_amount": 742.50,
            "shipping_cost": 0.00,
            "additional_charges": 0.00,
            "total_amount": 8992.50,
            "delivery_terms": "Delivery within 14 days",
            "warranty": "36 months comprehensive warranty",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "APPROVE",
            "reason": "Quotation includes an acceptable extended warranty and complete commercial information.",
            "issue": "NONE",
            "next_action": "CREATE_DRAFT_PO",
            "severity": "NONE"
        },
        "tags": ["extended_warranty", "valid_warranty"]
    },

    # ==========================================================
    # PQE-2026-015 — Net 45 payment terms
    # ==========================================================
    {
        "eval_id": "PQE-2026-015",
        "quotation": {
            "supplier_name": "PrimeTech Distribution Pte. Ltd.",
            "quotation_number": "QT-2026-1015",
            "quotation_date": "2026-09-20",
            "valid_until": "2026-10-20",
            "currency": "SGD",
            "items": [
                {
                    "description": "27-inch Business Monitor",
                    "model": "BM-27Q",
                    "quantity": 40,
                    "unit_price": 300.00,
                    "line_total": 12000.00
                }
            ],
            "subtotal": 12000.00,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 9.0,
            "tax_amount": 1080.00,
            "shipping_cost": 0.00,
            "additional_charges": 0.00,
            "total_amount": 13080.00,
            "delivery_terms": "Delivery within 14 days",
            "warranty": "24 months",
            "payment_terms": "Net 45"
        },
        "expected": {
            "decision": "APPROVE",
            "reason": "Quotation contains complete and acceptable commercial terms with correct calculations.",
            "issue": "NONE",
            "next_action": "CREATE_DRAFT_PO",
            "severity": "NONE"
        },
        "tags": ["net_45", "payment_terms"]
    },

    # ==========================================================
    # PQE-2026-016 — Split payment terms
    # ==========================================================
    {
        "eval_id": "PQE-2026-016",
        "quotation": {
            "supplier_name": "Horizon Enterprise Systems Pte. Ltd.",
            "quotation_number": "QT-2026-1016",
            "quotation_date": "2026-09-18",
            "valid_until": "2026-10-18",
            "currency": "SGD",
            "items": [
                {
                    "description": "Enterprise Wi-Fi Access Point",
                    "model": "AP-600",
                    "quantity": 20,
                    "unit_price": 650.00,
                    "line_total": 13000.00
                }
            ],
            "subtotal": 13000.00,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 9.0,
            "tax_amount": 1170.00,
            "shipping_cost": 0.00,
            "additional_charges": 0.00,
            "total_amount": 14170.00,
            "delivery_terms": "Delivery within 21 days",
            "warranty": "36 months",
            "payment_terms": "50% on order, 50% after delivery"
        },
        "expected": {
            "decision": "APPROVE",
            "reason": "Split payment terms are clearly defined and the quotation is otherwise complete and consistent.",
            "issue": "NONE",
            "next_action": "CREATE_DRAFT_PO",
            "severity": "NONE"
        },
        "tags": ["split_payment", "payment_terms"]
    },

    # ==========================================================
    # PQE-2026-017 — Three different product lines
    # ==========================================================
    {
        "eval_id": "PQE-2026-017",
        "quotation": {
            "supplier_name": "Apex Technology Pte. Ltd.",
            "quotation_number": "QT-2026-1017",
            "quotation_date": "2026-09-22",
            "valid_until": "2026-10-22",
            "currency": "SGD",
            "items": [
                {
                    "description": "Business Laptop",
                    "model": "BL-14X",
                    "quantity": 10,
                    "unit_price": 1300.00,
                    "line_total": 13000.00
                },
                {
                    "description": "27-inch Business Monitor",
                    "model": "BM-27Q",
                    "quantity": 10,
                    "unit_price": 350.00,
                    "line_total": 3500.00
                },
                {
                    "description": "USB-C Docking Station",
                    "model": "DS-100",
                    "quantity": 10,
                    "unit_price": 150.00,
                    "line_total": 1500.00
                }
            ],
            "subtotal": 18000.00,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 9.0,
            "tax_amount": 1620.00,
            "shipping_cost": 150.00,
            "additional_charges": 0.00,
            "total_amount": 19770.00,
            "delivery_terms": "All items delivered within 21 days",
            "warranty": "24 months",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "APPROVE",
            "reason": "All product lines, totals and commercial terms are complete and internally consistent.",
            "issue": "NONE",
            "next_action": "CREATE_DRAFT_PO",
            "severity": "NONE"
        },
        "tags": ["multiple_items", "three_products", "valid_calculation"]
    },

    # ==========================================================
    # PQE-2026-018 — Valid quotation close to expiry
    # ==========================================================
    {
        "eval_id": "PQE-2026-018",
        "quotation": {
            "supplier_name": "Nova Business Systems Pte. Ltd.",
            "quotation_number": "QT-2026-1018",
            "quotation_date": "2026-09-01",
            "valid_until": "2026-10-02",
            "currency": "SGD",
            "items": [
                {
                    "description": "Business Laptop",
                    "model": "BL-14X",
                    "quantity": 8,
                    "unit_price": 1450.00,
                    "line_total": 11600.00
                }
            ],
            "subtotal": 11600.00,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 9.0,
            "tax_amount": 1044.00,
            "shipping_cost": 0.00,
            "additional_charges": 0.00,
            "total_amount": 12644.00,
            "delivery_terms": "Delivery within 14 days",
            "warranty": "24 months",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "APPROVE",
            "reason": "Quotation remains valid on the reference date and all commercial information is complete.",
            "issue": "NONE",
            "next_action": "CREATE_DRAFT_PO",
            "severity": "NONE"
        },
        "tags": ["near_expiry", "valid_quotation"]
    },

    # ==========================================================
    # PQE-2026-019 — Long validity period
    # ==========================================================
    {
        "eval_id": "PQE-2026-019",
        "quotation": {
            "supplier_name": "Vertex Enterprise Supplies Pte. Ltd.",
            "quotation_number": "QT-2026-1019",
            "quotation_date": "2026-09-15",
            "valid_until": "2026-12-15",
            "currency": "SGD",
            "items": [
                {
                    "description": "Wireless Keyboard and Mouse Set",
                    "model": "KM-500",
                    "quantity": 50,
                    "unit_price": 75.00,
                    "line_total": 3750.00
                }
            ],
            "subtotal": 3750.00,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 9.0,
            "tax_amount": 337.50,
            "shipping_cost": 0.00,
            "additional_charges": 0.00,
            "total_amount": 4087.50,
            "delivery_terms": "Delivery within 7 days",
            "warranty": "12 months",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "APPROVE",
            "reason": "Quotation has a valid extended validity period and contains complete commercial information.",
            "issue": "NONE",
            "next_action": "CREATE_DRAFT_PO",
            "severity": "NONE"
        },
        "tags": ["long_validity", "valid_quotation"]
    },

    # ==========================================================
    # PQE-2026-020 — Tax-inclusive pricing
    # ==========================================================
    {
        "eval_id": "PQE-2026-020",
        "quotation": {
            "supplier_name": "Orion Office Technologies Pte. Ltd.",
            "quotation_number": "QT-2026-1020",
            "quotation_date": "2026-09-24",
            "valid_until": "2026-10-24",
            "currency": "SGD",
            "items": [
                {
                    "description": "USB-C Docking Station",
                    "model": "DS-100",
                    "quantity": 20,
                    "unit_price": 218.00,
                    "line_total": 4360.00
                }
            ],
            "subtotal": 4000.00,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 9.0,
            "tax_amount": 360.00,
            "shipping_cost": 0.00,
            "additional_charges": 0.00,
            "total_amount": 4360.00,
            "pricing_note": "Unit and line prices shown are inclusive of 9% GST.",
            "delivery_terms": "Delivery within 10 days",
            "warranty": "24 months",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "APPROVE",
            "reason": "Tax-inclusive pricing is explicitly disclosed and the underlying subtotal and GST reconcile correctly.",
            "issue": "NONE",
            "next_action": "CREATE_DRAFT_PO",
            "severity": "NONE"
        },
        "tags": ["tax_inclusive", "gst", "valid_calculation"]
    },

    # ==========================================================
    # PQE-2026-021 — Discount + shipping
    # ==========================================================
    {
        "eval_id": "PQE-2026-021",
        "quotation": {
            "supplier_name": "Summit Digital Solutions Pte. Ltd.",
            "quotation_number": "QT-2026-1021",
            "quotation_date": "2026-09-23",
            "valid_until": "2026-10-23",
            "currency": "SGD",
            "items": [
                {
                    "description": "27-inch Business Monitor",
                    "model": "BM-27Q",
                    "quantity": 20,
                    "unit_price": 450.00,
                    "line_total": 9000.00
                }
            ],
            "subtotal": 9000.00,
            "discount_percent": 5,
            "discount_amount": 450.00,
            "tax_percent": 9.0,
            "tax_amount": 769.50,
            "shipping_cost": 100.00,
            "additional_charges": 0.00,
            "total_amount": 9419.50,
            "delivery_terms": "Delivery within 14 days",
            "warranty": "24 months",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "APPROVE",
            "reason": "Discount, GST and shipping charges are clearly stated and correctly reflected in the final amount.",
            "issue": "NONE",
            "next_action": "CREATE_DRAFT_PO",
            "severity": "NONE"
        },
        "tags": ["discount", "shipping", "valid_calculation"]
    },

    # ==========================================================
    # PQE-2026-022 — Multiple items with discount
    # ==========================================================
    {
        "eval_id": "PQE-2026-022",
        "quotation": {
            "supplier_name": "Nexus Technology Pte. Ltd.",
            "quotation_number": "QT-2026-1022",
            "quotation_date": "2026-09-22",
            "valid_until": "2026-11-22",
            "currency": "SGD",
            "items": [
                {
                    "description": "Business Laptop",
                    "model": "BL-14X",
                    "quantity": 20,
                    "unit_price": 1200.00,
                    "line_total": 24000.00
                },
                {
                    "description": "USB-C Docking Station",
                    "model": "DS-100",
                    "quantity": 20,
                    "unit_price": 150.00,
                    "line_total": 3000.00
                }
            ],
            "subtotal": 27000.00,
            "discount_percent": 10,
            "discount_amount": 2700.00,
            "tax_percent": 9.0,
            "tax_amount": 2187.00,
            "shipping_cost": 0.00,
            "additional_charges": 0.00,
            "total_amount": 26487.00,
            "delivery_terms": "Delivery within 21 days",
            "warranty": "24 months",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "APPROVE",
            "reason": "Multi-item quotation and volume discount are correctly calculated with acceptable commercial terms.",
            "issue": "NONE",
            "next_action": "CREATE_DRAFT_PO",
            "severity": "NONE"
        },
        "tags": ["multiple_items", "discount", "valid_calculation"]
    },

    # ==========================================================
    # PQE-2026-023 — Minimum acceptable warranty
    # ==========================================================
    {
        "eval_id": "PQE-2026-023",
        "quotation": {
            "supplier_name": "PrimeTech Distribution Pte. Ltd.",
            "quotation_number": "QT-2026-1023",
            "quotation_date": "2026-09-24",
            "valid_until": "2026-10-24",
            "currency": "SGD",
            "items": [
                {
                    "description": "27-inch Business Monitor",
                    "model": "BM-27Q",
                    "quantity": 15,
                    "unit_price": 400.00,
                    "line_total": 6000.00
                }
            ],
            "subtotal": 6000.00,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 9.0,
            "tax_amount": 540.00,
            "shipping_cost": 0.00,
            "additional_charges": 0.00,
            "total_amount": 6540.00,
            "delivery_terms": "Delivery within 14 days",
            "warranty": "12 months",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "APPROVE",
            "reason": "Warranty meets the minimum procurement requirement and all other terms are valid.",
            "issue": "NONE",
            "next_action": "CREATE_DRAFT_PO",
            "severity": "NONE"
        },
        "tags": ["minimum_warranty", "boundary_case", "valid_quotation"]
    },

    # ==========================================================
    # PQE-2026-024 — Maximum preferred delivery period
    # ==========================================================
    {
        "eval_id": "PQE-2026-024",
        "quotation": {
            "supplier_name": "Horizon Enterprise Systems Pte. Ltd.",
            "quotation_number": "QT-2026-1024",
            "quotation_date": "2026-09-25",
            "valid_until": "2026-10-25",
            "currency": "SGD",
            "items": [
                {
                    "description": "Enterprise Wi-Fi Access Point",
                    "model": "AP-600",
                    "quantity": 30,
                    "unit_price": 500.00,
                    "line_total": 15000.00
                }
            ],
            "subtotal": 15000.00,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 9.0,
            "tax_amount": 1350.00,
            "shipping_cost": 0.00,
            "additional_charges": 0.00,
            "total_amount": 16350.00,
            "delivery_terms": "Delivery within 30 days",
            "warranty": "36 months",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "APPROVE",
            "reason": "Delivery is at the maximum preferred limit but remains within procurement policy.",
            "issue": "NONE",
            "next_action": "CREATE_DRAFT_PO",
            "severity": "NONE"
        },
        "tags": ["delivery_boundary", "30_days", "valid_quotation"]
    },

    # ==========================================================
    # PQE-2026-025 — Complex fully valid quotation
    # ==========================================================
    {
        "eval_id": "PQE-2026-025",
        "quotation": {
            "supplier_name": "Apex Technology Pte. Ltd.",
            "quotation_number": "QT-2026-1025",
            "quotation_date": "2026-09-25",
            "valid_until": "2026-11-25",
            "currency": "SGD",
            "items": [
                {
                    "description": "Business Laptop",
                    "model": "BL-14X",
                    "quantity": 30,
                    "unit_price": 1400.00,
                    "line_total": 42000.00
                },
                {
                    "description": "27-inch Business Monitor",
                    "model": "BM-27Q",
                    "quantity": 30,
                    "unit_price": 350.00,
                    "line_total": 10500.00
                },
                {
                    "description": "USB-C Docking Station",
                    "model": "DS-100",
                    "quantity": 30,
                    "unit_price": 150.00,
                    "line_total": 4500.00
                }
            ],
            "subtotal": 57000.00,
            "discount_percent": 10,
            "discount_amount": 5700.00,
            "tax_percent": 9.0,
            "tax_amount": 4617.00,
            "shipping_cost": 250.00,
            "additional_charges": 500.00,
            "total_amount": 56667.00,
            "delivery_terms": "Delivery and installation within 21 days",
            "warranty": "36 months",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "APPROVE",
            "reason": "Complex multi-item quotation is complete, internally consistent and within procurement policy.",
            "issue": "NONE",
            "next_action": "CREATE_DRAFT_PO",
            "severity": "NONE"
        },
        "tags": [
            "complex_quotation",
            "multiple_items",
            "discount",
            "shipping",
            "installation",
            "valid_calculation"
        ]
    }
]

print(f"APPROVE cases defined: {len(approve_cases)}")

assert len(approve_cases) == 25, \
    f"Expected 25 APPROVE cases, found {len(approve_cases)}"

assert all(
    case["expected"]["decision"] == "APPROVE"
    for case in approve_cases
), "One or more cases have an incorrect expected decision."

print("✓ All 25 APPROVE cases created successfully.")

APPROVE cases defined: 25
✓ All 25 APPROVE cases created successfully.


In [6]:
# ==========================================
# CELL 5: REVIEW QUOTATION CASES
# 10 quotations requiring supervisor review
# ==========================================

review_cases = [

    # ==========================================================
    # PQE-2026-026 — Value exceeds approval threshold
    # ==========================================================
    {
        "eval_id": "PQE-2026-026",
        "quotation": {
            "supplier_name": "Nova Business Systems Pte. Ltd.",
            "quotation_number": "QT-2026-1026",
            "quotation_date": "2026-09-24",
            "valid_until": "2026-10-24",
            "currency": "SGD",
            "items": [
                {
                    "description": "Business Laptop",
                    "model": "BL-14X",
                    "quantity": 80,
                    "unit_price": 1300.00,
                    "line_total": 104000.00
                }
            ],
            "subtotal": 104000.00,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 9.0,
            "tax_amount": 9360.00,
            "shipping_cost": 0.00,
            "additional_charges": 0.00,
            "total_amount": 113360.00,
            "delivery_terms": "Delivery within 21 days",
            "warranty": "24 months",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "REVIEW",
            "reason": "Quotation value exceeds the SGD 100,000 supervisor approval threshold.",
            "issue": "APPROVAL_THRESHOLD_EXCEEDED",
            "next_action": "SUPERVISOR_REVIEW",
            "severity": "MEDIUM"
        },
        "tags": [
            "high_value",
            "approval_threshold",
            "supervisor_authorization"
        ]
    },

    # ==========================================================
    # PQE-2026-027 — Unusually large order quantity
    # ==========================================================
    {
        "eval_id": "PQE-2026-027",
        "quotation": {
            "supplier_name": "Vertex Enterprise Supplies Pte. Ltd.",
            "quotation_number": "QT-2026-1027",
            "quotation_date": "2026-09-23",
            "valid_until": "2026-10-23",
            "currency": "SGD",
            "items": [
                {
                    "description": "Wireless Keyboard and Mouse Set",
                    "model": "KM-500",
                    "quantity": 1000,
                    "unit_price": 70.00,
                    "line_total": 70000.00
                }
            ],
            "subtotal": 70000.00,
            "discount_percent": 10,
            "discount_amount": 7000.00,
            "tax_percent": 9.0,
            "tax_amount": 5670.00,
            "shipping_cost": 500.00,
            "additional_charges": 0.00,
            "total_amount": 69170.00,
            "delivery_terms": "Delivery within 30 days",
            "warranty": "12 months",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "REVIEW",
            "reason": "The unusually large order quantity creates a significant procurement commitment requiring supervisor review.",
            "issue": "UNUSUALLY_LARGE_QUANTITY",
            "next_action": "SUPERVISOR_REVIEW",
            "severity": "MEDIUM"
        },
        "tags": [
            "large_quantity",
            "procurement_commitment",
            "supervisor_judgement"
        ]
    },

    # ==========================================================
    # PQE-2026-028 — Delivery exceeds preferred period
    # ==========================================================
    {
        "eval_id": "PQE-2026-028",
        "quotation": {
            "supplier_name": "Orion Office Technologies Pte. Ltd.",
            "quotation_number": "QT-2026-1028",
            "quotation_date": "2026-09-22",
            "valid_until": "2026-10-22",
            "currency": "SGD",
            "items": [
                {
                    "description": "Enterprise Wi-Fi Access Point",
                    "model": "AP-600",
                    "quantity": 20,
                    "unit_price": 600.00,
                    "line_total": 12000.00
                }
            ],
            "subtotal": 12000.00,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 9.0,
            "tax_amount": 1080.00,
            "shipping_cost": 0.00,
            "additional_charges": 0.00,
            "total_amount": 13080.00,
            "delivery_terms": "Delivery within 45 days",
            "warranty": "36 months",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "REVIEW",
            "reason": "The 45-day delivery period exceeds the preferred maximum of 30 days and requires business approval.",
            "issue": "EXTENDED_DELIVERY_PERIOD",
            "next_action": "SUPERVISOR_REVIEW",
            "severity": "MEDIUM"
        },
        "tags": [
            "extended_delivery",
            "delivery_risk",
            "policy_exception"
        ]
    },

    # ==========================================================
    # PQE-2026-029 — Warranty below preferred minimum
    # ==========================================================
    {
        "eval_id": "PQE-2026-029",
        "quotation": {
            "supplier_name": "Summit Digital Solutions Pte. Ltd.",
            "quotation_number": "QT-2026-1029",
            "quotation_date": "2026-09-24",
            "valid_until": "2026-10-24",
            "currency": "SGD",
            "items": [
                {
                    "description": "Business Laptop",
                    "model": "BL-14X",
                    "quantity": 20,
                    "unit_price": 1350.00,
                    "line_total": 27000.00
                }
            ],
            "subtotal": 27000.00,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 9.0,
            "tax_amount": 2430.00,
            "shipping_cost": 0.00,
            "additional_charges": 0.00,
            "total_amount": 29430.00,
            "delivery_terms": "Delivery within 14 days",
            "warranty": "6 months",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "REVIEW",
            "reason": "The offered warranty is below the preferred minimum of 12 months and requires supervisor acceptance.",
            "issue": "SHORT_WARRANTY",
            "next_action": "SUPERVISOR_REVIEW",
            "severity": "MEDIUM"
        },
        "tags": [
            "short_warranty",
            "warranty_risk",
            "policy_exception"
        ]
    },

    # ==========================================================
    # PQE-2026-030 — High upfront payment
    # ==========================================================
    {
        "eval_id": "PQE-2026-030",
        "quotation": {
            "supplier_name": "Nexus Technology Pte. Ltd.",
            "quotation_number": "QT-2026-1030",
            "quotation_date": "2026-09-25",
            "valid_until": "2026-10-25",
            "currency": "SGD",
            "items": [
                {
                    "description": "Business Laptop",
                    "model": "BL-14X",
                    "quantity": 25,
                    "unit_price": 1400.00,
                    "line_total": 35000.00
                }
            ],
            "subtotal": 35000.00,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 9.0,
            "tax_amount": 3150.00,
            "shipping_cost": 0.00,
            "additional_charges": 0.00,
            "total_amount": 38150.00,
            "delivery_terms": "Delivery within 21 days",
            "warranty": "24 months",
            "payment_terms": "80% upfront, 20% after delivery"
        },
        "expected": {
            "decision": "REVIEW",
            "reason": "The requirement for 80% upfront payment creates elevated financial exposure and requires supervisor approval.",
            "issue": "HIGH_UPFRONT_PAYMENT",
            "next_action": "SUPERVISOR_REVIEW",
            "severity": "HIGH"
        },
        "tags": [
            "upfront_payment",
            "financial_risk",
            "payment_terms"
        ]
    },

    # ==========================================================
    # PQE-2026-031 — Full advance payment
    # ==========================================================
    {
        "eval_id": "PQE-2026-031",
        "quotation": {
            "supplier_name": "PrimeTech Distribution Pte. Ltd.",
            "quotation_number": "QT-2026-1031",
            "quotation_date": "2026-09-24",
            "valid_until": "2026-10-24",
            "currency": "SGD",
            "items": [
                {
                    "description": "27-inch Business Monitor",
                    "model": "BM-27Q",
                    "quantity": 40,
                    "unit_price": 375.00,
                    "line_total": 15000.00
                }
            ],
            "subtotal": 15000.00,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 9.0,
            "tax_amount": 1350.00,
            "shipping_cost": 0.00,
            "additional_charges": 0.00,
            "total_amount": 16350.00,
            "delivery_terms": "Delivery within 14 days",
            "warranty": "24 months",
            "payment_terms": "100% payment before shipment"
        },
        "expected": {
            "decision": "REVIEW",
            "reason": "Full payment is required before shipment, creating elevated supplier and financial risk.",
            "issue": "FULL_ADVANCE_PAYMENT",
            "next_action": "SUPERVISOR_REVIEW",
            "severity": "HIGH"
        },
        "tags": [
            "advance_payment",
            "financial_risk",
            "supplier_risk"
        ]
    },

    # ==========================================================
    # PQE-2026-032 — Unusually high shipping cost
    # ==========================================================
    {
        "eval_id": "PQE-2026-032",
        "quotation": {
            "supplier_name": "Horizon Enterprise Systems Pte. Ltd.",
            "quotation_number": "QT-2026-1032",
            "quotation_date": "2026-09-25",
            "valid_until": "2026-10-25",
            "currency": "SGD",
            "items": [
                {
                    "description": "USB-C Docking Station",
                    "model": "DS-100",
                    "quantity": 20,
                    "unit_price": 180.00,
                    "line_total": 3600.00
                }
            ],
            "subtotal": 3600.00,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 9.0,
            "tax_amount": 324.00,
            "shipping_cost": 1500.00,
            "additional_charges": 0.00,
            "total_amount": 5424.00,
            "delivery_terms": "Delivery within 10 days",
            "warranty": "24 months",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "REVIEW",
            "reason": "The shipping charge is unusually high relative to the value of the goods and requires commercial review.",
            "issue": "HIGH_SHIPPING_COST",
            "next_action": "SUPERVISOR_REVIEW",
            "severity": "MEDIUM"
        },
        "tags": [
            "high_shipping",
            "commercial_risk",
            "cost_review"
        ]
    },

    # ==========================================================
    # PQE-2026-033 — Foreign currency quotation
    # ==========================================================
    {
        "eval_id": "PQE-2026-033",
        "quotation": {
            "supplier_name": "Apex Technology Pte. Ltd.",
            "quotation_number": "QT-2026-1033",
            "quotation_date": "2026-09-25",
            "valid_until": "2026-10-25",
            "currency": "USD",
            "items": [
                {
                    "description": "Business Laptop",
                    "model": "BL-14X",
                    "quantity": 20,
                    "unit_price": 950.00,
                    "line_total": 19000.00
                }
            ],
            "subtotal": 19000.00,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 0.0,
            "tax_amount": 0.00,
            "shipping_cost": 300.00,
            "additional_charges": 0.00,
            "total_amount": 19300.00,
            "delivery_terms": "Delivery within 21 days",
            "warranty": "24 months",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "REVIEW",
            "reason": "The quotation is denominated in USD rather than the procurement base currency SGD and requires currency and financial review.",
            "issue": "FOREIGN_CURRENCY",
            "next_action": "SUPERVISOR_REVIEW",
            "severity": "MEDIUM"
        },
        "tags": [
            "foreign_currency",
            "usd",
            "currency_risk"
        ]
    },

    # ==========================================================
    # PQE-2026-034 — Very short quotation validity
    # ==========================================================
    {
        "eval_id": "PQE-2026-034",
        "quotation": {
            "supplier_name": "Nova Business Systems Pte. Ltd.",
            "quotation_number": "QT-2026-1034",
            "quotation_date": "2026-09-30",
            "valid_until": "2026-10-01",
            "currency": "SGD",
            "items": [
                {
                    "description": "Enterprise Wi-Fi Access Point",
                    "model": "AP-600",
                    "quantity": 25,
                    "unit_price": 550.00,
                    "line_total": 13750.00
                }
            ],
            "subtotal": 13750.00,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 9.0,
            "tax_amount": 1237.50,
            "shipping_cost": 0.00,
            "additional_charges": 0.00,
            "total_amount": 14987.50,
            "delivery_terms": "Delivery within 14 days",
            "warranty": "36 months",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "REVIEW",
            "reason": "The quotation remains valid but expires within one day, requiring time-sensitive supervisor review.",
            "issue": "SHORT_VALIDITY_PERIOD",
            "next_action": "SUPERVISOR_REVIEW",
            "severity": "MEDIUM"
        },
        "tags": [
            "short_validity",
            "time_sensitive",
            "commercial_review"
        ]
    },

    # ==========================================================
    # PQE-2026-035 — High cancellation/restocking charge
    # ==========================================================
    {
        "eval_id": "PQE-2026-035",
        "quotation": {
            "supplier_name": "Vertex Enterprise Supplies Pte. Ltd.",
            "quotation_number": "QT-2026-1035",
            "quotation_date": "2026-09-25",
            "valid_until": "2026-10-25",
            "currency": "SGD",
            "items": [
                {
                    "description": "Business Laptop",
                    "model": "BL-14X",
                    "quantity": 30,
                    "unit_price": 1350.00,
                    "line_total": 40500.00
                }
            ],
            "subtotal": 40500.00,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 9.0,
            "tax_amount": 3645.00,
            "shipping_cost": 0.00,
            "additional_charges": 0.00,
            "total_amount": 44145.00,
            "delivery_terms": "Delivery within 21 days",
            "warranty": "24 months",
            "payment_terms": "Net 30",
            "special_terms": "Order cancellation after confirmation is subject to a non-refundable 30% restocking charge."
        },
        "expected": {
            "decision": "REVIEW",
            "reason": "The 30% non-refundable cancellation charge creates a significant contractual commitment requiring supervisor review.",
            "issue": "HIGH_CANCELLATION_FEE",
            "next_action": "SUPERVISOR_REVIEW",
            "severity": "HIGH"
        },
        "tags": [
            "cancellation_fee",
            "contractual_risk",
            "commercial_terms"
        ]
    }
]

print(f"REVIEW cases defined: {len(review_cases)}")

assert len(review_cases) == 10, \
    f"Expected 10 REVIEW cases, found {len(review_cases)}"

assert all(
    case["expected"]["decision"] == "REVIEW"
    for case in review_cases
), "One or more cases have an incorrect expected decision."

print("✓ All 10 REVIEW cases created successfully.")

REVIEW cases defined: 10
✓ All 10 REVIEW cases created successfully.


In [7]:
# ==========================================
# CELL 6: REQUEST CLARIFICATION CASES
# 7 quotations requiring supplier clarification
# ==========================================

clarification_cases = [

    # ==========================================================
    # PQE-2026-036 — Incorrect line total
    # ==========================================================
    {
        "eval_id": "PQE-2026-036",
        "quotation": {
            "supplier_name": "Orion Office Technologies Pte. Ltd.",
            "quotation_number": "QT-2026-1036",
            "quotation_date": "2026-09-25",
            "valid_until": "2026-10-25",
            "currency": "SGD",
            "items": [
                {
                    "description": "27-inch Business Monitor",
                    "model": "BM-27Q",
                    "quantity": 20,
                    "unit_price": 500.00,
                    "line_total": 12000.00
                }
            ],
            "subtotal": 12000.00,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 9.0,
            "tax_amount": 1080.00,
            "shipping_cost": 0.00,
            "additional_charges": 0.00,
            "total_amount": 13080.00,
            "delivery_terms": "Delivery within 14 days",
            "warranty": "24 months",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "REQUEST_CLARIFICATION",
            "reason": "The stated line total does not match the quantity multiplied by the unit price.",
            "issue": "LINE_TOTAL_MISMATCH",
            "next_action": "CONTACT_SUPPLIER",
            "severity": "HIGH"
        },
        "tags": [
            "calculation_error",
            "line_total",
            "quantity_price_mismatch"
        ]
    },

    # ==========================================================
    # PQE-2026-037 — Incorrect subtotal
    # ==========================================================
    {
        "eval_id": "PQE-2026-037",
        "quotation": {
            "supplier_name": "Summit Digital Solutions Pte. Ltd.",
            "quotation_number": "QT-2026-1037",
            "quotation_date": "2026-09-24",
            "valid_until": "2026-10-24",
            "currency": "SGD",
            "items": [
                {
                    "description": "Business Laptop",
                    "model": "BL-14X",
                    "quantity": 10,
                    "unit_price": 1200.00,
                    "line_total": 12000.00
                },
                {
                    "description": "USB-C Docking Station",
                    "model": "DS-100",
                    "quantity": 10,
                    "unit_price": 150.00,
                    "line_total": 1500.00
                }
            ],
            "subtotal": 14000.00,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 9.0,
            "tax_amount": 1260.00,
            "shipping_cost": 0.00,
            "additional_charges": 0.00,
            "total_amount": 15260.00,
            "delivery_terms": "Delivery within 14 days",
            "warranty": "24 months",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "REQUEST_CLARIFICATION",
            "reason": "The stated subtotal does not equal the sum of the individual line totals.",
            "issue": "SUBTOTAL_MISMATCH",
            "next_action": "CONTACT_SUPPLIER",
            "severity": "HIGH"
        },
        "tags": [
            "calculation_error",
            "subtotal",
            "multiple_items"
        ]
    },

    # ==========================================================
    # PQE-2026-038 — Incorrect GST calculation
    # ==========================================================
    {
        "eval_id": "PQE-2026-038",
        "quotation": {
            "supplier_name": "Nexus Technology Pte. Ltd.",
            "quotation_number": "QT-2026-1038",
            "quotation_date": "2026-09-25",
            "valid_until": "2026-10-25",
            "currency": "SGD",
            "items": [
                {
                    "description": "Enterprise Wi-Fi Access Point",
                    "model": "AP-600",
                    "quantity": 20,
                    "unit_price": 500.00,
                    "line_total": 10000.00
                }
            ],
            "subtotal": 10000.00,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 9.0,
            "tax_amount": 1200.00,
            "shipping_cost": 0.00,
            "additional_charges": 0.00,
            "total_amount": 11200.00,
            "delivery_terms": "Delivery within 14 days",
            "warranty": "36 months",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "REQUEST_CLARIFICATION",
            "reason": "The stated GST amount is inconsistent with the quoted 9% tax rate and subtotal.",
            "issue": "INCORRECT_TAX_CALCULATION",
            "next_action": "CONTACT_SUPPLIER",
            "severity": "HIGH"
        },
        "tags": [
            "calculation_error",
            "gst",
            "tax_mismatch"
        ]
    },

    # ==========================================================
    # PQE-2026-039 — Incorrect discount calculation
    # ==========================================================
    {
        "eval_id": "PQE-2026-039",
        "quotation": {
            "supplier_name": "PrimeTech Distribution Pte. Ltd.",
            "quotation_number": "QT-2026-1039",
            "quotation_date": "2026-09-23",
            "valid_until": "2026-10-23",
            "currency": "SGD",
            "items": [
                {
                    "description": "Business Laptop",
                    "model": "BL-14X",
                    "quantity": 20,
                    "unit_price": 1500.00,
                    "line_total": 30000.00
                }
            ],
            "subtotal": 30000.00,
            "discount_percent": 10,
            "discount_amount": 2000.00,
            "tax_percent": 9.0,
            "tax_amount": 2520.00,
            "shipping_cost": 0.00,
            "additional_charges": 0.00,
            "total_amount": 30520.00,
            "delivery_terms": "Delivery within 21 days",
            "warranty": "24 months",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "REQUEST_CLARIFICATION",
            "reason": "The stated discount amount does not correspond to the quoted 10% discount rate.",
            "issue": "INCORRECT_DISCOUNT_CALCULATION",
            "next_action": "CONTACT_SUPPLIER",
            "severity": "HIGH"
        },
        "tags": [
            "calculation_error",
            "discount",
            "discount_mismatch"
        ]
    },

    # ==========================================================
    # PQE-2026-040 — Contradictory delivery terms
    # ==========================================================
    {
        "eval_id": "PQE-2026-040",
        "quotation": {
            "supplier_name": "Horizon Enterprise Systems Pte. Ltd.",
            "quotation_number": "QT-2026-1040",
            "quotation_date": "2026-09-25",
            "valid_until": "2026-10-25",
            "currency": "SGD",
            "items": [
                {
                    "description": "27-inch Business Monitor",
                    "model": "BM-27Q",
                    "quantity": 25,
                    "unit_price": 400.00,
                    "line_total": 10000.00
                }
            ],
            "subtotal": 10000.00,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 9.0,
            "tax_amount": 900.00,
            "shipping_cost": 0.00,
            "additional_charges": 0.00,
            "total_amount": 10900.00,
            "delivery_terms": "Delivery within 14 days",
            "warranty": "24 months",
            "payment_terms": "Net 30",
            "special_terms": "Standard delivery lead time for this order is 30 days."
        },
        "expected": {
            "decision": "REQUEST_CLARIFICATION",
            "reason": "The quotation contains conflicting delivery periods of 14 days and 30 days.",
            "issue": "CONFLICTING_DELIVERY_TERMS",
            "next_action": "CONTACT_SUPPLIER",
            "severity": "MEDIUM"
        },
        "tags": [
            "contradictory_information",
            "delivery_terms",
            "ambiguity"
        ]
    },

    # ==========================================================
    # PQE-2026-041 — Missing warranty information
    # ==========================================================
    {
        "eval_id": "PQE-2026-041",
        "quotation": {
            "supplier_name": "Apex Technology Pte. Ltd.",
            "quotation_number": "QT-2026-1041",
            "quotation_date": "2026-09-24",
            "valid_until": "2026-10-24",
            "currency": "SGD",
            "items": [
                {
                    "description": "Business Laptop",
                    "model": "BL-14X",
                    "quantity": 15,
                    "unit_price": 1350.00,
                    "line_total": 20250.00
                }
            ],
            "subtotal": 20250.00,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 9.0,
            "tax_amount": 1822.50,
            "shipping_cost": 0.00,
            "additional_charges": 0.00,
            "total_amount": 22072.50,
            "delivery_terms": "Delivery within 14 days",
            "warranty": None,
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "REQUEST_CLARIFICATION",
            "reason": "The quotation does not specify warranty coverage for the supplied equipment.",
            "issue": "MISSING_WARRANTY",
            "next_action": "CONTACT_SUPPLIER",
            "severity": "MEDIUM"
        },
        "tags": [
            "missing_information",
            "warranty",
            "incomplete_terms"
        ]
    },

    # ==========================================================
    # PQE-2026-042 — Currency not specified
    # ==========================================================
    {
        "eval_id": "PQE-2026-042",
        "quotation": {
            "supplier_name": "Nova Business Systems Pte. Ltd.",
            "quotation_number": "QT-2026-1042",
            "quotation_date": "2026-09-25",
            "valid_until": "2026-10-25",
            "currency": None,
            "items": [
                {
                    "description": "Enterprise Wi-Fi Access Point",
                    "model": "AP-600",
                    "quantity": 10,
                    "unit_price": 650.00,
                    "line_total": 6500.00
                }
            ],
            "subtotal": 6500.00,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 9.0,
            "tax_amount": 585.00,
            "shipping_cost": 100.00,
            "additional_charges": 0.00,
            "total_amount": 7185.00,
            "delivery_terms": "Delivery within 14 days",
            "warranty": "36 months",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "REQUEST_CLARIFICATION",
            "reason": "The quotation does not identify the currency used for its prices and totals.",
            "issue": "MISSING_CURRENCY",
            "next_action": "CONTACT_SUPPLIER",
            "severity": "HIGH"
        },
        "tags": [
            "missing_information",
            "currency",
            "pricing_ambiguity"
        ]
    }
]

print(f"REQUEST_CLARIFICATION cases defined: {len(clarification_cases)}")

assert len(clarification_cases) == 7, \
    f"Expected 7 REQUEST_CLARIFICATION cases, found {len(clarification_cases)}"

assert all(
    case["expected"]["decision"] == "REQUEST_CLARIFICATION"
    for case in clarification_cases
), "One or more cases have an incorrect expected decision."

print("✓ All 7 REQUEST_CLARIFICATION cases created successfully.")

REQUEST_CLARIFICATION cases defined: 7
✓ All 7 REQUEST_CLARIFICATION cases created successfully.


In [8]:
# ==========================================
# CELL 7: REJECT QUOTATION CASES
# 8 inputs that should not proceed
# ==========================================

reject_cases = [

    # ==========================================================
    # PQE-2026-043 — Invoice uploaded instead of quotation
    # ==========================================================
    {
        "eval_id": "PQE-2026-043",
        "quotation": {
            "document_type": "INVOICE",
            "supplier_name": "Vertex Enterprise Supplies Pte. Ltd.",
            "document_number": "INV-2026-7842",
            "document_date": "2026-09-24",
            "currency": "SGD",
            "items": [
                {
                    "description": "Business Laptop",
                    "model": "BL-14X",
                    "quantity": 10,
                    "unit_price": 1300.00,
                    "line_total": 13000.00
                }
            ],
            "subtotal": 13000.00,
            "tax_percent": 9.0,
            "tax_amount": 1170.00,
            "total_amount": 14170.00,
            "payment_status": "Payment Due"
        },
        "expected": {
            "decision": "REJECT",
            "reason": "The submitted document is an invoice rather than a supplier quotation.",
            "issue": "INVALID_DOCUMENT_TYPE",
            "next_action": "STOP_PROCESSING",
            "severity": "HIGH"
        },
        "tags": [
            "wrong_document",
            "invoice",
            "invalid_document_type"
        ]
    },

    # ==========================================================
    # PQE-2026-044 — Purchase order uploaded
    # ==========================================================
    {
        "eval_id": "PQE-2026-044",
        "quotation": {
            "document_type": "PURCHASE_ORDER",
            "supplier_name": "Apex Technology Pte. Ltd.",
            "document_number": "PO-2026-4581",
            "document_date": "2026-09-22",
            "currency": "SGD",
            "items": [
                {
                    "description": "27-inch Business Monitor",
                    "model": "BM-27Q",
                    "quantity": 20,
                    "unit_price": 400.00,
                    "line_total": 8000.00
                }
            ],
            "subtotal": 8000.00,
            "tax_percent": 9.0,
            "tax_amount": 720.00,
            "total_amount": 8720.00,
            "order_status": "Issued"
        },
        "expected": {
            "decision": "REJECT",
            "reason": "The submitted document is a purchase order rather than a supplier quotation.",
            "issue": "INVALID_DOCUMENT_TYPE",
            "next_action": "STOP_PROCESSING",
            "severity": "HIGH"
        },
        "tags": [
            "wrong_document",
            "purchase_order",
            "invalid_document_type"
        ]
    },

    # ==========================================================
    # PQE-2026-045 — Unrelated business document
    # ==========================================================
    {
        "eval_id": "PQE-2026-045",
        "quotation": {
            "document_type": "COMPANY_PROFILE",
            "supplier_name": "Summit Digital Solutions Pte. Ltd.",
            "document_title": "Corporate Capabilities and Services Profile",
            "document_date": "2026-09-10",
            "content_summary": (
                "Company background, service portfolio, certifications, "
                "office locations and contact information."
            )
        },
        "expected": {
            "decision": "REJECT",
            "reason": "The submitted document is a company profile and does not contain a supplier quotation.",
            "issue": "UNRELATED_DOCUMENT",
            "next_action": "STOP_PROCESSING",
            "severity": "HIGH"
        },
        "tags": [
            "wrong_document",
            "unrelated_document",
            "no_quotation"
        ]
    },

    # ==========================================================
    # PQE-2026-046 — Quotation expired several years ago
    # ==========================================================
    {
        "eval_id": "PQE-2026-046",
        "quotation": {
            "document_type": "SUPPLIER_QUOTATION",
            "supplier_name": "Orion Office Technologies Pte. Ltd.",
            "quotation_number": "QT-2022-0421",
            "quotation_date": "2022-05-10",
            "valid_until": "2022-06-10",
            "currency": "SGD",
            "items": [
                {
                    "description": "Business Laptop",
                    "model": "BL-14X",
                    "quantity": 20,
                    "unit_price": 1100.00,
                    "line_total": 22000.00
                }
            ],
            "subtotal": 22000.00,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 7.0,
            "tax_amount": 1540.00,
            "shipping_cost": 0.00,
            "additional_charges": 0.00,
            "total_amount": 23540.00,
            "delivery_terms": "Delivery within 14 days",
            "warranty": "24 months",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "REJECT",
            "reason": "The quotation expired several years before the current procurement assessment and is no longer commercially valid.",
            "issue": "SEVERELY_EXPIRED_QUOTATION",
            "next_action": "STOP_PROCESSING",
            "severity": "HIGH"
        },
        "tags": [
            "expired_quotation",
            "stale_document",
            "invalid_validity"
        ]
    },

    # ==========================================================
    # PQE-2026-047 — Implausible future-dated quotation
    # ==========================================================
    {
        "eval_id": "PQE-2026-047",
        "quotation": {
            "document_type": "SUPPLIER_QUOTATION",
            "supplier_name": "Nexus Technology Pte. Ltd.",
            "quotation_number": "QT-2035-9001",
            "quotation_date": "2035-01-15",
            "valid_until": "2035-02-15",
            "currency": "SGD",
            "items": [
                {
                    "description": "Enterprise Wi-Fi Access Point",
                    "model": "AP-600",
                    "quantity": 20,
                    "unit_price": 600.00,
                    "line_total": 12000.00
                }
            ],
            "subtotal": 12000.00,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 9.0,
            "tax_amount": 1080.00,
            "shipping_cost": 0.00,
            "additional_charges": 0.00,
            "total_amount": 13080.00,
            "delivery_terms": "Delivery within 14 days",
            "warranty": "24 months",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "REJECT",
            "reason": "The quotation is dated implausibly far in the future relative to the assessment date.",
            "issue": "INVALID_FUTURE_DATE",
            "next_action": "STOP_PROCESSING",
            "severity": "HIGH"
        },
        "tags": [
            "future_date",
            "suspicious_document",
            "invalid_date"
        ]
    },

    # ==========================================================
    # PQE-2026-048 — Supplier identity absent
    # ==========================================================
    {
        "eval_id": "PQE-2026-048",
        "quotation": {
            "document_type": "SUPPLIER_QUOTATION",
            "supplier_name": None,
            "quotation_number": "QT-2026-1048",
            "quotation_date": "2026-09-25",
            "valid_until": "2026-10-25",
            "currency": "SGD",
            "items": [
                {
                    "description": "USB-C Docking Station",
                    "model": "DS-100",
                    "quantity": 20,
                    "unit_price": 180.00,
                    "line_total": 3600.00
                }
            ],
            "subtotal": 3600.00,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 9.0,
            "tax_amount": 324.00,
            "shipping_cost": 0.00,
            "additional_charges": 0.00,
            "total_amount": 3924.00,
            "delivery_terms": "Delivery within 14 days",
            "warranty": "24 months",
            "payment_terms": "Net 30",
            "supplier_contact": None
        },
        "expected": {
            "decision": "REJECT",
            "reason": "The quotation contains no identifiable supplier or supplier contact information, so its source cannot be established.",
            "issue": "UNIDENTIFIABLE_SUPPLIER",
            "next_action": "STOP_PROCESSING",
            "severity": "CRITICAL"
        },
        "tags": [
            "missing_supplier",
            "source_unverifiable",
            "invalid_quotation"
        ]
    },

    # ==========================================================
    # PQE-2026-049 — Unreadable/corrupted quotation
    # ==========================================================
    {
        "eval_id": "PQE-2026-049",
        "quotation": {
            "document_type": "SUPPLIER_QUOTATION",
            "supplier_name": "PrimeTech Distribution Pte. Ltd.",
            "quotation_number": None,
            "quotation_date": None,
            "valid_until": None,
            "currency": None,
            "items": [],
            "subtotal": None,
            "tax_percent": None,
            "tax_amount": None,
            "total_amount": None,
            "delivery_terms": None,
            "warranty": None,
            "payment_terms": None,
            "extraction_status": "UNREADABLE",
            "extraction_note": (
                "Document content is severely corrupted or illegible. "
                "No reliable commercial information could be extracted."
            )
        },
        "expected": {
            "decision": "REJECT",
            "reason": "The document is unreadable and does not contain sufficient reliable information for procurement assessment.",
            "issue": "UNREADABLE_DOCUMENT",
            "next_action": "STOP_PROCESSING",
            "severity": "CRITICAL"
        },
        "tags": [
            "unreadable_document",
            "extraction_failure",
            "insufficient_information"
        ]
    },

    # ==========================================================
    # PQE-2026-050 — Quotation for unrelated goods
    # ==========================================================
    {
        "eval_id": "PQE-2026-050",
        "quotation": {
            "document_type": "SUPPLIER_QUOTATION",
            "supplier_name": "Horizon Enterprise Systems Pte. Ltd.",
            "quotation_number": "QT-2026-1050",
            "quotation_date": "2026-09-25",
            "valid_until": "2026-10-25",
            "currency": "SGD",
            "procurement_request": "Business laptops and accessories",
            "items": [
                {
                    "description": "Industrial Refrigeration Unit",
                    "model": "IRU-5000",
                    "quantity": 2,
                    "unit_price": 18000.00,
                    "line_total": 36000.00
                }
            ],
            "subtotal": 36000.00,
            "discount_percent": 0,
            "discount_amount": 0.00,
            "tax_percent": 9.0,
            "tax_amount": 3240.00,
            "shipping_cost": 500.00,
            "additional_charges": 0.00,
            "total_amount": 39740.00,
            "delivery_terms": "Delivery within 21 days",
            "warranty": "24 months",
            "payment_terms": "Net 30"
        },
        "expected": {
            "decision": "REJECT",
            "reason": "The quotation is for goods unrelated to the stated procurement requirement.",
            "issue": "PROCUREMENT_SCOPE_MISMATCH",
            "next_action": "STOP_PROCESSING",
            "severity": "HIGH"
        },
        "tags": [
            "wrong_goods",
            "scope_mismatch",
            "irrelevant_quotation"
        ]
    }
]

print(f"REJECT cases defined: {len(reject_cases)}")

assert len(reject_cases) == 8, \
    f"Expected 8 REJECT cases, found {len(reject_cases)}"

assert all(
    case["expected"]["decision"] == "REJECT"
    for case in reject_cases
), "One or more cases have an incorrect expected decision."

print("✓ All 8 REJECT cases created successfully.")

REJECT cases defined: 8
✓ All 8 REJECT cases created successfully.


In [9]:
# ==========================================
# CELL 8: COMBINE & VALIDATE DATASET
# ==========================================

from collections import Counter

# Combine all quotation cases
dataset = (
    approve_cases
    + review_cases
    + clarification_cases
    + reject_cases
)

print(f"Total cases combined: {len(dataset)}")


# ------------------------------------------
# 1. Validate total number of cases
# ------------------------------------------

assert len(dataset) == 50, \
    f"Expected 50 cases, found {len(dataset)}"


# ------------------------------------------
# 2. Validate unique evaluation IDs
# ------------------------------------------

eval_ids = [case["eval_id"] for case in dataset]

assert len(eval_ids) == len(set(eval_ids)), \
    "Duplicate eval_id detected."


# ------------------------------------------
# 3. Validate sequential ID format
# ------------------------------------------

expected_ids = [
    f"PQE-2026-{i:03d}"
    for i in range(1, 51)
]

assert eval_ids == expected_ids, \
    "Evaluation IDs are missing, duplicated, or out of sequence."


# ------------------------------------------
# 4. Validate required top-level fields
# ------------------------------------------

required_fields = {
    "eval_id",
    "quotation",
    "expected",
    "tags"
}

for case in dataset:

    missing_fields = required_fields - set(case.keys())

    assert not missing_fields, \
        f"{case['eval_id']} missing fields: {missing_fields}"


# ------------------------------------------
# 5. Validate expected-output fields
# ------------------------------------------

required_expected_fields = {
    "decision",
    "reason",
    "issue",
    "next_action",
    "severity"
}

for case in dataset:

    missing_fields = (
        required_expected_fields
        - set(case["expected"].keys())
    )

    assert not missing_fields, \
        f"{case['eval_id']} expected output missing: {missing_fields}"


# ------------------------------------------
# 6. Validate decision distribution
# ------------------------------------------

decision_counts = Counter(
    case["expected"]["decision"]
    for case in dataset
)

expected_distribution = {
    "APPROVE": 25,
    "REVIEW": 10,
    "REQUEST_CLARIFICATION": 7,
    "REJECT": 8
}

assert dict(decision_counts) == expected_distribution, \
    f"Incorrect decision distribution: {decision_counts}"


# ------------------------------------------
# 7. Validate decision → action mapping
# ------------------------------------------

expected_actions = {
    "APPROVE": "CREATE_DRAFT_PO",
    "REVIEW": "SUPERVISOR_REVIEW",
    "REQUEST_CLARIFICATION": "CONTACT_SUPPLIER",
    "REJECT": "STOP_PROCESSING"
}

for case in dataset:

    decision = case["expected"]["decision"]
    action = case["expected"]["next_action"]

    assert action == expected_actions[decision], \
        (
            f"{case['eval_id']} has invalid action: "
            f"{decision} → {action}"
        )


# ------------------------------------------
# 8. Validate allowed severity levels
# ------------------------------------------

allowed_severities = {
    "NONE",
    "LOW",
    "MEDIUM",
    "HIGH",
    "CRITICAL"
}

for case in dataset:

    severity = case["expected"]["severity"]

    assert severity in allowed_severities, \
        f"{case['eval_id']} has invalid severity: {severity}"


# ------------------------------------------
# Validation summary
# ------------------------------------------

print("\n✓ Dataset validation successful")
print("✓ 50 total cases")
print("✓ All evaluation IDs unique and sequential")
print("✓ Required fields present")
print("✓ Expected-output schema valid")
print("✓ Decision/action mappings valid")

print("\nDecision distribution:")

for decision, expected_count in expected_distribution.items():
    actual_count = decision_counts[decision]
    print(f"  {decision:<22} {actual_count}")

Total cases combined: 50

✓ Dataset validation successful
✓ 50 total cases
✓ All evaluation IDs unique and sequential
✓ Required fields present
✓ Expected-output schema valid
✓ Decision/action mappings valid

Decision distribution:
  APPROVE                25
  REVIEW                 10
  REQUEST_CLARIFICATION  7
  REJECT                 8


In [10]:
# ==========================================
# CELL 9: SAVE DATASET TO JSON
# ==========================================

# Save the validated dataset
with open(DATA_FILE, "w", encoding="utf-8") as f:
    json.dump(
        dataset,
        f,
        indent=2,
        ensure_ascii=False
    )

print(f"✓ Dataset saved successfully")
print(f"File: {DATA_FILE}")


# ------------------------------------------
# Reload file to verify successful write
# ------------------------------------------

with open(DATA_FILE, "r", encoding="utf-8") as f:
    saved_dataset = json.load(f)

assert len(saved_dataset) == 50, \
    f"Saved file contains {len(saved_dataset)} cases instead of 50."

assert saved_dataset == dataset, \
    "Saved dataset does not match the generated dataset."

print(f"✓ File verified successfully")
print(f"✓ {len(saved_dataset)} evaluation cases stored")


# ------------------------------------------
# Preview first evaluation case
# ------------------------------------------

print("\nSample evaluation case:\n")

print(
    json.dumps(
        saved_dataset[0],
        indent=2,
        ensure_ascii=False
    )
)

✓ Dataset saved successfully
File: /content/drive/MyDrive/QuoteGuard - AI Procurement/data/data.json
✓ File verified successfully
✓ 50 evaluation cases stored

Sample evaluation case:

{
  "eval_id": "PQE-2026-001",
  "quotation": {
    "supplier_name": "Apex Technology Pte. Ltd.",
    "quotation_number": "QT-2026-1001",
    "quotation_date": "2026-09-20",
    "valid_until": "2026-10-20",
    "currency": "SGD",
    "items": [
      {
        "description": "Business Laptop",
        "model": "BL-14X",
        "quantity": 20,
        "unit_price": 1250.0,
        "line_total": 25000.0
      }
    ],
    "subtotal": 25000.0,
    "discount_percent": 0,
    "discount_amount": 0.0,
    "tax_percent": 9.0,
    "tax_amount": 2250.0,
    "shipping_cost": 0.0,
    "additional_charges": 0.0,
    "total_amount": 27250.0,
    "delivery_terms": "Delivery within 14 days",
    "warranty": "24 months",
    "payment_terms": "Net 30"
  },
  "expected": {
    "decision": "APPROVE",
    "reason": "Quotati